# Phase Diagrams: Damped Harmonic Oscillator

Compare phase-space trajectories for underdamped, critically damped, and overdamped motion. The horizontal axis is position and the vertical axis is velocity. All three cases use the same initial position and velocity, with the natural angular frequency fixed at $\\omega_0 = 1$.

Change the initial conditions and the damping coefficients to see how the trajectories differ. This notebook uses `ipywidgets` and standard inline Matplotlib output.

## Model

The equation of motion is

$$
\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2 x = 0.
$$

For this notebook, $\omega_0=1$. The critical case has $\gamma=\omega_0=1$. The underdamped case has $\gamma<1$, and the overdamped case has $\gamma>1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print('NumPy:', np.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('ipywidgets:', widgets.__version__)
print('Matplotlib backend:', plt.get_backend())

In [ ]:
# Fixed natural angular frequency and time array
w0 = 1.0
t = np.linspace(0.0, 15.0, 1800)

# Shared initial conditions
spos = widgets.FloatSlider(
    value=5.0, min=-5.0, max=5.0, step=0.1,
    description='Initial position', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
svel = widgets.FloatSlider(
    value=0.0, min=-5.0, max=5.0, step=0.1,
    description='Initial velocity', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Damping coefficients for the underdamped and overdamped cases
sgam_under = widgets.FloatSlider(
    value=0.5, min=0.0, max=0.99, step=0.01,
    description='Underdamped γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
sgam_over = widgets.FloatSlider(
    value=1.5, min=1.01, max=5.0, step=0.01,
    description='Overdamped γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

def phase_solutions(x0, v0, gamma_under, gamma_over):
    # Critical damping: repeated root r = -w0
    critical_x = (x0 + (v0 + w0*x0)*t) * np.exp(-w0*t)
    critical_v = (v0 - w0*(v0 + w0*x0)*t) * np.exp(-w0*t)

    # Underdamped case: gamma < w0
    wd = np.sqrt(w0**2 - gamma_under**2)
    under_x = np.exp(-gamma_under*t) * (
        x0*np.cos(wd*t) + ((v0 + gamma_under*x0)/wd)*np.sin(wd*t)
    )
    under_v = np.exp(-gamma_under*t) * (
        v0*np.cos(wd*t) - ((gamma_under*v0 + w0**2*x0)/wd)*np.sin(wd*t)
    )

    # Overdamped case: two distinct real roots
    q = np.sqrt(gamma_over**2 - w0**2)
    r1 = -gamma_over + q
    r2 = -gamma_over - q
    c1 = (v0 - r2*x0)/(r1 - r2)
    c2 = (r1*x0 - v0)/(r1 - r2)
    over_x = c1*np.exp(r1*t) + c2*np.exp(r2*t)
    over_v = r1*c1*np.exp(r1*t) + r2*c2*np.exp(r2*t)

    return (critical_x, critical_v), (under_x, under_v), (over_x, over_v)

def plot_phase_diagrams(x0, v0, gamma_under, gamma_over):
    critical, under, over = phase_solutions(x0, v0, gamma_under, gamma_over)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(critical[0], critical[1], color='blue', linewidth=2,
            label='Critical damping (γ = 1)')
    ax.plot(under[0], under[1], color='green', linewidth=1.8,
            label=f'Underdamped (γ = {gamma_under:.2f})')
    ax.plot(over[0], over[1], color='red', linewidth=1.8,
            label=f'Overdamped (γ = {gamma_over:.2f})')

    # Mark the common initial state
    ax.plot([x0], [v0], marker='o', color='black', markersize=6,
            linestyle='None', label='Initial state')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('Position, x')
    ax.set_ylabel('Velocity, v')
    ax.set_title('Phase Diagrams for Three Damping Regimes')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)

plot_output = widgets.interactive_output(
    plot_phase_diagrams,
    {
        'x0': spos,
        'v0': svel,
        'gamma_under': sgam_under,
        'gamma_over': sgam_over
    }
)

reset_button = widgets.Button(description='Reset')
def reset_values(button):
    spos.value = 5.0
    svel.value = 0.0
    sgam_under.value = 0.5
    sgam_over.value = 1.5
reset_button.on_click(reset_values)

controls = widgets.VBox([spos, svel, sgam_under, sgam_over, reset_button])
display(controls, plot_output)

## Questions to investigate

1. Which trajectory spirals toward the origin, and why?
2. Which trajectories approach the origin without oscillating?
3. How does increasing the overdamped damping coefficient change the trajectory?
4. What happens to the underdamped trajectory as its damping coefficient approaches 1?
5. All three curves begin at the same point. Why do their paths differ afterward?

The curves are plotted only over a finite time interval. Depending on the damping coefficient, a trajectory may approach the origin so closely that its later portion is difficult to distinguish from the axes.